# Store Popular MovieLens Movies in MySQL with Spark JDBC

This notebook follows `D304-MovieLens.ipynb`: it reads MovieLens data from HDFS, keeps movies with at least **100 ratings** and an **average rating of 3.5 or higher**, then writes the result to MySQL through JDBC.

The MySQL username and password are read only from `MYSQL_USERNAME` and `MYSQL_PASSWORD`. They are not stored in the notebook.

## 1. Install MySQL Connector/J 8 in WSL

Run this in a **WSL terminal before starting Jupyter or Spark**. Connector/J 8.4 is compatible with MySQL Server 8 and uses driver class `com.mysql.cj.jdbc.Driver`.

```bash
MYSQL_JDBC_VERSION=8.4.0
MYSQL_JDBC_JAR="mysql-connector-j-${MYSQL_JDBC_VERSION}.jar"

wget -O "/tmp/$MYSQL_JDBC_JAR" \
  "https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/${MYSQL_JDBC_VERSION}/${MYSQL_JDBC_JAR}"

test -s "/tmp/$MYSQL_JDBC_JAR"
cp "/tmp/$MYSQL_JDBC_JAR" "$SPARK_HOME/jars/$MYSQL_JDBC_JAR"
ls -lh "$SPARK_HOME/jars/$MYSQL_JDBC_JAR"
```

Because the JAR is placed in `$SPARK_HOME/jars`, Spark loads it when a new driver and workers start. Stop any existing Spark session, restart the standalone master/worker if used, and restart the notebook kernel after installing it. Do not keep multiple Connector/J versions in this directory.

Check for duplicates:

```bash
find "$SPARK_HOME/jars" -maxdepth 1 -name 'mysql-connector-j-*.jar' -print
```

## 2. Prepare MySQL and environment variables

Create the target database once. The MySQL account must be allowed to connect from Spark and must have privileges on `movielens`. For a local teaching installation, connect as an administrator and run: 

```sql
CREATE DATABASE IF NOT EXISTS movielens
  CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_ai_ci;
GRANT ALL PRIVILEGES ON movielens.* TO 'your_user'@'localhost';
```

Set credentials in the **same WSL shell that launches Jupyter**. `read -s` prevents the password from being echoed or saved in shell history.

```bash
export MYSQL_USERNAME='your_user'
read -rsp 'MySQL password: ' MYSQL_PASSWORD && echo
export MYSQL_PASSWORD

# Optional overrides; these are the notebook defaults:
export MYSQL_HOST='localhost'
export MYSQL_PORT='3306'
export MYSQL_DATABASE='movielens'

jupyter lab
```

If MySQL runs on Windows rather than inside WSL, set `MYSQL_HOST` to the Windows host IP visible from WSL and configure MySQL `bind-address`, the user host, and Windows Firewall appropriately.

## 3. Start Spark

Start HDFS and, when using standalone mode, the Spark master and worker as described in `D304-MovieLens.ipynb`. The MovieLens CSV files must already exist under `/user/$USER/movielens/movies` and `/user/$USER/movielens/ratings`.

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

required_env = ("MYSQL_USERNAME", "MYSQL_PASSWORD")
missing_env = [name for name in required_env if not os.environ.get(name)]
if missing_env:
    raise RuntimeError(
        "Set these environment variables before starting Jupyter: "
        + ", ".join(missing_env)
    )

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D30-MovieLens-MySQL-JDBC")
    .master(master_url)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master :", spark.sparkContext.master)

## 4. Read MovieLens data from HDFS with explicit schemas

In [ ]:
from pyspark.sql.types import (
    DoubleType, IntegerType, LongType, StringType, StructType,
)

movie_schema = (
    StructType()
    .add("movieId", IntegerType(), nullable=False)
    .add("title", StringType(), nullable=False)
    .add("genres", StringType(), nullable=True)
)
rating_schema = (
    StructType()
    .add("userId", IntegerType(), nullable=False)
    .add("movieId", IntegerType(), nullable=False)
    .add("rating", DoubleType(), nullable=False)
    .add("timestamp", LongType(), nullable=False)
)

hdfs_user = os.environ["USER"]
movies_path = f"hdfs:///user/{hdfs_user}/movielens/movies"
ratings_path = f"hdfs:///user/{hdfs_user}/movielens/ratings"

movie_df = spark.read.option("header", True).schema(movie_schema).csv(movies_path)
rating_df = spark.read.option("header", True).schema(rating_schema).csv(ratings_path)

print("Movies :", movie_df.count())
print("Ratings:", rating_df.count())

## 5. Calculate popular movies

In [ ]:
from pyspark.sql.functions import avg, col, count, current_timestamp, desc

popular_movies_df = (
    rating_df
    .groupBy("movieId")
    .agg(
        avg("rating").alias("avg_rating"),
        count("userId").alias("total_ratings"),
    )
    .filter((col("total_ratings") >= 100) & (col("avg_rating") >= 3.5))
    .join(movie_df, on="movieId", how="inner")
    .select("movieId", "title", "genres", "avg_rating", "total_ratings")
    .withColumn("loaded_at", current_timestamp())
    .orderBy(desc("total_ratings"), desc("avg_rating"))
    .cache()
)

popular_movies_df.show(20, truncate=False)
print("Popular movies:", popular_movies_df.count())

## 6. Configure JDBC

Only non-secret connection details are printed. `useSSL=false` is suitable for this local teaching setup; use TLS and certificate validation for a remote or production database.

In [ ]:
mysql_host = os.environ.get("MYSQL_HOST", "localhost")
mysql_port = os.environ.get("MYSQL_PORT", "3306")
mysql_database = os.environ.get("MYSQL_DATABASE", "movielens")
mysql_table = os.environ.get("MYSQL_TABLE", "popular_movies")

jdbc_url = (
    f"jdbc:mysql://{mysql_host}:{mysql_port}/{mysql_database}"
    "?useSSL=false&allowPublicKeyRetrieval=true&serverTimezone=UTC"
)
jdbc_properties = {
    "user": os.environ["MYSQL_USERNAME"],
    "password": os.environ["MYSQL_PASSWORD"],
    "driver": "com.mysql.cj.jdbc.Driver",
}

print(f"Target: mysql://{mysql_host}:{mysql_port}/{mysql_database}/{mysql_table}")

## 7. Write to MySQL

`overwrite` makes repeated lesson runs idempotent by replacing the table. `truncate=true` asks Spark to truncate a compatible existing MySQL table instead of dropping and recreating it. The small result is reduced to a few JDBC writers to avoid opening unnecessary database connections.

In [ ]:
(
    popular_movies_df.coalesce(2).write
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", mysql_table)
    .option("user", jdbc_properties["user"])
    .option("password", jdbc_properties["password"])
    .option("driver", jdbc_properties["driver"])
    .option("batchsize", "1000")
    .option("truncate", "true")
    .mode("overwrite")
    .save()
)
print(f"Wrote table: {mysql_database}.{mysql_table}")

## 8. Read the MySQL table back and verify it

In [ ]:
saved_df = spark.read.jdbc(
    url=jdbc_url,
    table=mysql_table,
    properties=jdbc_properties,
)

source_count = popular_movies_df.count()
saved_count = saved_df.count()
assert saved_count == source_count, (
    f"Row-count mismatch: Spark={source_count}, MySQL={saved_count}"
)

saved_df.orderBy(desc("total_ratings"), desc("avg_rating")).show(20, truncate=False)
print("Verified rows:", saved_count)

You can also verify from the MySQL client:

```sql
USE movielens;
SELECT COUNT(*) FROM popular_movies;
SELECT * FROM popular_movies ORDER BY total_ratings DESC, avg_rating DESC LIMIT 20;
```

## 9. Stop Spark

In [ ]:
popular_movies_df.unpersist()
spark.stop()
print("Spark session stopped.")